In [ ]:
#@title ⚙️ Setup — choose how to provide the inputs (RUN ME FIRST)
#@markdown Pick how the input files for this step are provided:
#@markdown - **Mobile mode**: the files are downloaded automatically from the GitHub repo — no manual upload, works on phones/tablets.
#@markdown - **Manual upload**: you upload the files yourself in the cells below.
input_mode = "Mobile mode (auto-download from repo)" #@param ["Mobile mode (auto-download from repo)", "Manual upload"]

import os, urllib.request
REPO_RAW = "https://raw.githubusercontent.com/biochorl/Nanobody_de_novo_design/main"

# Inputs needed by THIS step.  (some links are FACSIMILE placeholders until the files
# are added to the repo — they will start working once they are committed)
INPUT_FILES = [('/content/AntiFold_Best_Designs.zip', '/Intermediate_inputs/AntiFold_Best_Designs.zip', 'facsimile')]

def fetch_inputs(files):
    ok = 0
    for dst, rel, kind in files:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        url = REPO_RAW + rel
        try:
            urllib.request.urlretrieve(url, dst)
            print(f"✅ {os.path.basename(dst):<28} ← {url}  [{kind}]")
            ok += 1
        except Exception as e:
            tag = "facsimile not in repo yet" if kind == "facsimile" else "ERROR"
            print(f"⚠️  {os.path.basename(dst):<28} could not be downloaded ({tag}): {e}")
    return ok

if input_mode.startswith("Mobile"):
    print("📱 Mobile mode — downloading this step's inputs from the repo:\n")
    n = fetch_inputs(INPUT_FILES)
    print(f"\n→ {n}/{len(INPUT_FILES)} file(s) ready in /content. "
          "Skip the manual-upload cells and point the path fields to these files.")
else:
    print("🖱️ Manual mode — upload your files in the cells below as usual.")


# **ESMFold Structural Validation**
## Overview
This notebook uses the ESMFold API to predict the 3D structure of the redesigned nanobody-antigen complexes and visually compare them with the IgGM generated models. You can upload the `AntiFold_Best_Designs.zip`, refold them, and interactively visualize the alignment with original designs using py3Dmol.

> 🔑 You need a free **BioHub API token** (paste it in the folding cell). Get it on the main site or directly at https://biohub.ai/developer-console/api-keys

In [ ]:
# @title 1. Install Dependencies
!pip install -q py3Dmol biopython
# Official BioHub ESM SDK (needed to decode the ESMFold2 response into a valid mmCIF)
!pip install -q "esm @ git+https://github.com/Biohub/esm.git@main"


In [ ]:
# @title 2. Provide AntiFold Best Designs (auto in Mobile mode, else upload)
import os, zipfile
from google.colab import files

output_dir = "/content/PIPPack_designs"
os.makedirs(output_dir, exist_ok=True)
auto = "/content/AntiFold_Best_Designs.zip"

if os.path.exists(auto):
    print(f"📂 Auto mode: using {auto}")
    with zipfile.ZipFile(auto) as z:
        z.extractall(output_dir)
else:
    print("Upload AntiFold_Best_Designs.zip:")
    uploaded = files.upload()
    for fn in uploaded:
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn) as z:
                z.extractall(output_dir)
            print(f"Extracted {fn} to {output_dir}")
        elif fn.endswith('.pdb') or fn.endswith('.fasta'):
            os.rename(fn, os.path.join(output_dir, fn))
            print(f"Saved {fn}")

pdbs = [f for f in os.listdir(output_dir) if f.endswith('.pdb')]
print(f"\nFound {len(pdbs)} PDB files ready for validation.")
if os.path.exists(os.path.join(output_dir, "top1_sequences.fasta")):
    print("Found top1_sequences.fasta containing AntiFold sequence redesigns.")


In [ ]:
# @title 3. Fold Sequences with ESMFold2 (BioHub)
# @markdown Paste your BioHub API token below. Get one at: https://biohub.ai/developer-console/api-keys
biohub_token = "" # @param {type:"string"}
# @markdown **Model** — use the non-fast model for higher quality on difficult folds.
esm_model = "esmfold2-2026-05" # @param ["esmfold2-2026-05", "esmfold2-fast-2026-05"]
# @markdown **num_loops (cycles)** — trunk refinement loops; more = higher quality, slower (max 20).
num_loops = 20 # @param {type:"slider", min:0, max:20, step:1}
# @markdown **num_sampling_steps** — diffusion solver steps; higher = better/slower (max 100).
num_sampling_steps = 50 # @param {type:"slider", min:1, max:100, step:1}

import os, time, string
from Bio import PDB
from Bio.SeqUtils import seq1

from esm.sdk import esmfold2_client
from esm.sdk.api import FoldingConfig, ESMProteinError
from esm.utils.structure import input_builder

output_dir = "/content/PIPPack_designs"
esm_output_dir = "/content/ESMFold_designs"
os.makedirs(esm_output_dir, exist_ok=True)

def get_sequences_for_folding(pdb_file, fasta_file):
    # Parse AntiFold redesigned sequences
    antifold_seqs = {}
    if os.path.exists(fasta_file):
        with open(fasta_file, 'r') as f:
            for block in f.read().split('>'):
                if block.strip():
                    parts = block.strip().split('\n')
                    name = parts[0].split()[0]
                    seq = ''.join(parts[1:])
                    antifold_seqs[name] = seq

    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('struct', pdb_file)
    sequences = []
    for model in structure:
        for chain in model:
            seq = ""
            for residue in chain:
                if PDB.is_aa(residue, standard=True):
                    seq += seq1(residue.resname)
            if seq:
                sequences.append(seq)

    pdb_name = os.path.basename(pdb_file)
    if pdb_name in antifold_seqs and sequences:
        original_seq = sequences[0]
        new_seq = antifold_seqs[pdb_name]
        print(f"  -> Merging AntiFold seq (len {len(new_seq)}) with Antigen seq (len {len(sequences[1]) if len(sequences)>1 else 0})")
        sequences[0] = new_seq

    return sequences

def _to_float(x):
    try:
        if hasattr(x, "detach"):
            x = x.detach().cpu()
        import numpy as _np
        return float(_np.asarray(x))
    except Exception:
        return None

def _mean_plddt(plddt):
    if plddt is None:
        return None
    import numpy as _np
    arr = plddt.detach().cpu().numpy() if hasattr(plddt, "detach") else _np.asarray(plddt)
    arr = arr.astype(float)
    if arr.size == 0:
        return None
    m = float(arr.mean())
    return m * 100.0 if arr.max() <= 1.0 else m  # normalize 0-1 -> 0-100

def fold_sequence(client, config, sequences, save_path):
    # nanobody first ('H'), antigen last ('A'); fall back to A,B,... for other chain counts
    if len(sequences) == 2:
        chain_ids = ['H', 'A']
    else:
        chain_ids = [string.ascii_uppercase[i] for i in range(len(sequences))]

    prot_inputs = [input_builder.ProteinInput(id=cid, sequence=seq)
                   for cid, seq in zip(chain_ids, sequences)]
    spi = input_builder.StructurePredictionInput(sequences=prot_inputs)

    try:
        result = client.fold_all_atom(spi, config=config)
    except Exception as e:
        print(f"BioHub API error: {e}")
        return None

    if isinstance(result, ESMProteinError):
        print(f"BioHub API error: {result}")
        return None

    # The SDK decodes the response into a complex; export a proper mmCIF
    cif_str = result.complex.to_mmcif()
    if not cif_str.lstrip().startswith("data_"):
        print("⚠️ Unexpected mmCIF content returned by the API.")
        return None

    with open(save_path, "w") as f:
        f.write(cif_str)

    # --- confidence metrics returned by ESMFold2 ---
    metrics = {
        "mean_pLDDT": _mean_plddt(result.plddt),
        "pTM": _to_float(result.ptm),
        "ipTM": _to_float(result.iptm),  # interface confidence (nanobody-antigen)
    }
    return metrics

pdbs = sorted([f for f in os.listdir(output_dir) if f.endswith('.pdb')])
fasta_path = os.path.join(output_dir, "top1_sequences.fasta")

if not biohub_token:
    print("❌ Please provide your BioHub token in the form above and run the cell again.")
else:
    client = esmfold2_client(model=esm_model, url="https://biohub.ai", token=biohub_token)
    config = FoldingConfig(num_loops=num_loops, num_sampling_steps=num_sampling_steps,
                          include_pae=True, include_pair_chains_iptm=True)
    print(f"Using model '{esm_model}' with num_loops={num_loops}, num_sampling_steps={num_sampling_steps}.")

    import json
    summary = []
    for pdb in pdbs:
        print(f"Processing {pdb}...")
        seqs = get_sequences_for_folding(os.path.join(output_dir, pdb), fasta_path)
        total_len = sum(len(s) for s in seqs)
        print(f"Sequence length: {total_len} AA across {len(seqs)} chain(s).")

        save_path = os.path.join(esm_output_dir, pdb.replace('.pdb', '_esmfold2.cif'))
        metrics = fold_sequence(client, config, seqs, save_path)
        if metrics is not None:
            def _fmt(v, nd):
                return f"{v:.{nd}f}" if isinstance(v, (int, float)) else "n/a"
            print(f"✅ Folded: {save_path}")
            print(f"   📊 mean pLDDT: {_fmt(metrics['mean_pLDDT'], 1)} | "
                  f"pTM: {_fmt(metrics['pTM'], 3)} | "
                  f"ipTM (interface): {_fmt(metrics['ipTM'], 3)}")
            with open(save_path.replace('_esmfold2.cif', '_metrics.json'), 'w') as f:
                json.dump(metrics, f, indent=2)
            summary.append((pdb, metrics))
        else:
            print(f"❌ Failed to fold {pdb}")
        time.sleep(1)

    if summary:
        print("\n================ Confidence summary ================")
        print(f"{'design':<24}{'mean pLDDT':>12}{'pTM':>8}{'ipTM':>8}")
        for name, m in summary:
            mp = f"{m['mean_pLDDT']:.1f}" if isinstance(m['mean_pLDDT'], (int, float)) else "n/a"
            pt = f"{m['pTM']:.3f}" if isinstance(m['pTM'], (int, float)) else "n/a"
            it = f"{m['ipTM']:.3f}" if isinstance(m['ipTM'], (int, float)) else "n/a"
            print(f"{name:<24}{mp:>12}{pt:>8}{it:>8}")
        print("ipTM is the key metric for the nanobody-antigen interface (higher is better; >0.7 is good).")


In [ ]:
# @title 4. Visualize & compare (Design vs ESMFold refolded nanobody)
# @markdown Pick a design. The ESMFold model is superposed on the **antigen**, then the RMSD is
# @markdown measured on the **nanobody** (lower = the refolded nanobody is closer to the design).
import os, io
import numpy as np
import py3Dmol
import ipywidgets as widgets
from IPython.display import display, HTML
from Bio.PDB import PDBParser, MMCIFParser, Superimposer
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from Bio.PDB.mmcifio import MMCIFIO

def load_cif_tolerant(struct_id, path):
    """Parse ESMFold mmCIF even if it omits the '_atom_site.occupancy' column."""
    parser = MMCIFParser(QUIET=True)
    try:
        return parser.get_structure(struct_id, path)
    except KeyError:
        d = MMCIF2Dict(path); n = len(d["_atom_site.Cartn_x"])
        d.setdefault("_atom_site.occupancy", ["1.00"] * n)
        d.setdefault("_atom_site.B_iso_or_equiv", ["0.00"] * n)
        parser._mmcif_dict = d; parser._build_structure(struct_id)
        return parser._structure_builder.get_structure()

def split_nb_ag(struct):
    """Return (nb_chain, nb_CA, ag_chain, ag_CA). Antigen = longest protein chain,
    nanobody = shortest — robust to chain relabeling (ESMFold renames chains)."""
    prot = []
    for ch in struct.get_chains():
        cas = [a for a in ch.get_atoms() if a.id == 'CA']
        if cas:
            prot.append((ch, cas))
    prot.sort(key=lambda x: len(x[1]))
    return prot[0][0], prot[0][1], prot[-1][0], prot[-1][1]

output_dir = "/content/PIPPack_designs"
esm_output_dir = "/content/ESMFold_designs"

pdbs = sorted(f for f in os.listdir(output_dir) if f.endswith('.pdb'))
if not pdbs:
    print("No PDBs found. Run the previous cells first.")
else:
    dropdown = widgets.Dropdown(options=pdbs, description='Model:')

    def align_and_show(selected_pdb):
        ref_path = os.path.join(output_dir, selected_pdb)
        esm_path = os.path.join(esm_output_dir, selected_pdb.replace('.pdb', '_esmfold2.cif'))
        if not os.path.exists(esm_path):
            print(f"ESMFold2 output not found for {selected_pdb}")
            return

        ref_struct = PDBParser(QUIET=True).get_structure('ref', ref_path)
        esm_struct = load_cif_tolerant('esm', esm_path)
        try:
            ref_nb_ch, ref_nb, ref_ag_ch, ref_ag = split_nb_ag(ref_struct)
            esm_nb_ch, esm_nb, esm_ag_ch, esm_ag = split_nb_ag(esm_struct)
        except Exception:
            print("Need a 2-chain complex (nanobody + antigen)."); return

        # 1) superpose on the ANTIGEN (longest chain); move the whole ESMFold model
        m = min(len(ref_ag), len(esm_ag))
        si = Superimposer(); si.set_atoms(ref_ag[:m], esm_ag[:m]); si.apply(esm_struct.get_atoms())
        # 2) RMSD on the NANOBODY in that antigen-aligned frame (no re-superposition)
        k = min(len(ref_nb), len(esm_nb))
        nb_rmsd = float(np.sqrt(((np.array([a.coord for a in ref_nb[:k]]) -
                                  np.array([a.coord for a in esm_nb[:k]])) ** 2).sum() / k)) if k else float('nan')
        print(f"Antigen superposition RMSD = {si.rms:.2f} Å  (ref chain {ref_ag_ch.id} ↔ ESMFold chain {esm_ag_ch.id}).")
        print(f"➡️  Nanobody RMSD (antigen-aligned) = {nb_rmsd:.2f} Å   — lower means closer to the design.")

        buf = io.StringIO(); w = MMCIFIO(); w.set_structure(esm_struct); w.save(buf)
        aligned_esm_str = buf.getvalue()
        ref_str = open(ref_path).read()

        display(HTML(
            '<div style="font-family:sans-serif;font-size:14px;margin:10px 0;line-height:1.9">'
            '<b>Legend</b>&nbsp;&nbsp;'
            '<span style="color:#2fb56e;font-size:20px;vertical-align:middle">●</span> Design nanobody (reference)&nbsp;&nbsp;&nbsp;'
            '<span style="color:#1d3fb5;font-size:20px;vertical-align:middle">●</span> Refolded nanobody (ESMFold)&nbsp;&nbsp;&nbsp;'
            '<span style="color:#9aa3ad;font-size:20px;vertical-align:middle">●</span> Antigen (alignment frame)'
            f'&nbsp;&nbsp;&nbsp;|&nbsp;&nbsp; Nanobody RMSD: <b>{nb_rmsd:.2f} Å</b></div>'))

        view = py3Dmol.view(width=820, height=600)
        view.addModel(ref_str, 'pdb')           # model 0 = design (reference)
        view.addModel(aligned_esm_str, 'cif')   # model 1 = ESMFold (aligned)
        view.setStyle({}, {})                    # clear everything
        # reference: antigen (grey, context) + design nanobody (green) as cartoon
        view.setStyle({'model': 0, 'chain': ref_ag_ch.id}, {'cartoon': {'color': '#9aa3ad', 'opacity': 0.5}})
        view.setStyle({'model': 0, 'chain': ref_nb_ch.id}, {'cartoon': {'color': '#2fb56e'}})
        # ESMFold: nanobody (blue) as cartoon; its antigen is left hidden to avoid clutter
        view.setStyle({'model': 1, 'chain': esm_nb_ch.id}, {'cartoon': {'color': '#1d3fb5'}})
        view.zoomTo({'model': 0, 'chain': ref_nb_ch.id})
        view.show()

    widgets.interact(align_and_show, selected_pdb=dropdown);


In [ ]:
#@title 5. Download final comparative models (design vs ESMFold, antigen-aligned)
#@markdown Saves, for every design, the **reference design** and the **antigen-aligned ESMFold model**,
#@markdown plus a CSV with the nanobody RMSD — all zipped and downloaded.
import os, glob, shutil
import numpy as np
from Bio.PDB import PDBParser, MMCIFParser, Superimposer, PDBIO
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from google.colab import files

def _load_cif(struct_id, path):
    p = MMCIFParser(QUIET=True)
    try:
        return p.get_structure(struct_id, path)
    except KeyError:
        d = MMCIF2Dict(path); n = len(d["_atom_site.Cartn_x"])
        d.setdefault("_atom_site.occupancy", ["1.00"]*n)
        d.setdefault("_atom_site.B_iso_or_equiv", ["0.00"]*n)
        p._mmcif_dict = d; p._build_structure(struct_id)
        return p._structure_builder.get_structure()

output_dir = "/content/PIPPack_designs"
esm_output_dir = "/content/ESMFold_designs"
comp_dir = "/content/Comparative_models"
os.makedirs(comp_dir, exist_ok=True)

def _split(struct):
    prot=[]
    for c in struct.get_chains():
        cas=[a for a in c.get_atoms() if a.id=='CA']
        if cas: prot.append((c,cas))
    prot.sort(key=lambda x: len(x[1]))
    return prot[0][1], prot[-1][1]   # (nanobody_CA, antigen_CA)  shortest=nanobody, longest=antigen

rows = []
for pdb in sorted(f for f in os.listdir(output_dir) if f.endswith('.pdb')):
    esm_path = os.path.join(esm_output_dir, pdb.replace('.pdb', '_esmfold2.cif'))
    if not os.path.exists(esm_path):
        continue
    ref = PDBParser(QUIET=True).get_structure('ref', os.path.join(output_dir, pdb))
    esm = _load_cif('esm', esm_path)
    ref_nb, ref_ag = _split(ref)
    esm_nb, esm_ag = _split(esm)
    m=min(len(ref_ag),len(esm_ag)); si=Superimposer(); si.set_atoms(ref_ag[:m],esm_ag[:m]); si.apply(esm.get_atoms())
    k=min(len(ref_nb),len(esm_nb))
    nb_rmsd=float(np.sqrt(((np.array([a.coord for a in ref_nb[:k]])-np.array([a.coord for a in esm_nb[:k]]))**2).sum()/k)) if k else float('nan')
    rows.append((pdb, nb_rmsd, si.rms))
    io_w=PDBIO(); io_w.set_structure(esm); io_w.save(os.path.join(comp_dir, pdb.replace('.pdb','_ESMFold_aligned.pdb')))
    shutil.copy(os.path.join(output_dir, pdb), os.path.join(comp_dir, pdb.replace('.pdb','_design.pdb')))

if not rows:
    print("No ESMFold outputs found — run the folding cell first.")
else:
    with open(os.path.join(comp_dir,'comparison_metrics.csv'),'w') as f:
        f.write("design,nanobody_RMSD_A,antigen_superposition_RMSD_A\n")
        for name,nb,ag in rows: f.write(f"{name},{nb:.3f},{ag:.3f}\n")
    print("Nanobody RMSD (after antigen alignment):")
    for name,nb,ag in rows: print(f"  {name}: {nb:.2f} A")
    archive = shutil.make_archive("/content/Comparative_models", 'zip', comp_dir)
    print(f"\n✅ {len(rows)} comparative model pair(s) zipped. Downloading...")
    files.download(archive)


---
### ✅ Step complete — disconnect before the next notebook

To free the GPU/CPU for the next step, **disconnect and delete this runtime**:
`Runtime → Disconnect and delete runtime` (or `Manage sessions → Terminate`).

⬅️ **[Back to the main workflow](https://biochorl.github.io/Nanobody_de_novo_design/)**
